# Primordial Soup: replaying the tape of life, at scale

Runs the evolution simulation headless for many seeds under each setting below, then compares how often each
stage emerges (replicators → cells → proteins → DNA → photosynthesis → Great Oxidation → eukaryotes → sex →
multicellular → complex organisms), when it happens, and what the first cells' genomes looked like.

Simulation code: commit `bd3c1ff`, built 2026-10-15.

Works on any Jupyter server (Linux or macOS) and on Kaggle. **Before running:** if there is no Node.js 18+, the setup
cell downloads it from nodejs.org, and it installs pandas / scipy / matplotlib if they are missing, so it needs internet
access the first time (on Kaggle: switch *Internet* on). It uses the CPUs and memory your container is actually allowed.
Results are saved after every world (`primordial-soup-runs/results-<RUN_TAG>.jsonl`), so if the kernel stops, run all
cells again and it carries on where it left off. Change `RUN_TAG` to start a fresh experiment.


In [ ]:
# ---- Experiment settings -------------------------------------------------------------------------
# Each entry is a set of overrides for js/config.js (CFG). The first one is the baseline.
CONFIGS = {
    "baseline":      {},
    # Protocells draw in nucleotides from ~2 µm around them (2x the copying inside), and need 3 completed
    # self-copies (not 1) before becoming a cell, which keeps independent origins of life as rare as now.
    "pool_balanced": {"VES_NT_REACH": 4, "CELL_COPIES": 3},
}
TICKS = 70_000            # simulated ticks per world (late stages typically arrive at 40k-70k)
SEEDS_PER_CONFIG = None   # None: chosen from the number of cores (one or two full waves)
FIRST_SEED = 1
WORKERS = None            # None: as many as the cores and memory allow
RUN_TAG = "pool1"         # results go to primordial-soup-runs/results-<RUN_TAG>.jsonl


In [ ]:
# ---- Environment: cores, memory, Python packages, Node.js -----------------------------------------
import os, sys, json, shutil, subprocess, platform, tarfile, urllib.request, base64, time, importlib.util

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.join(os.getcwd(), "primordial-soup-runs")
os.makedirs(WORK, exist_ok=True)
SIM = os.path.join(WORK, "primordial-soup")

missing = [m for m in ("pandas", "scipy", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    print("Installing", *missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def read(path):
    try:
        with open(path) as f:
            return f.read().strip()
    except OSError:
        return None

def usable_cores():
    # The CPUs this process may run on, and any container (cgroup) quota, which os.cpu_count() ignores.
    n = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    q = read("/sys/fs/cgroup/cpu.max")                          # cgroup v2: "<quota> <period>" or "max ..."
    if q and not q.startswith("max"):
        quota, period = map(int, q.split()[:2]); n = min(n, max(1, quota // period))
    q, p = read("/sys/fs/cgroup/cpu/cpu.cfs_quota_us"), read("/sys/fs/cgroup/cpu/cpu.cfs_period_us")  # v1
    if q and p and int(q) > 0:
        n = min(n, max(1, int(q) // int(p)))
    return n

def free_gb():
    # Memory still available, within the container's limit if it has one.
    avail = None
    for line in (read("/proc/meminfo") or "").splitlines():
        if line.startswith("MemAvailable:"):
            avail = int(line.split()[1]) / 1e6
    if avail is None:
        try:
            import psutil; avail = psutil.virtual_memory().available / 1e9
        except ImportError:
            avail = 8.0
    for lim, cur in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                     ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        L, C = read(lim), read(cur)
        if L and C and L.isdigit() and int(L) < 1 << 60:
            avail = min(avail, (int(L) - int(C)) / 1e9)
    return avail

cores, mem = usable_cores(), free_gb()
GB_PER_WORLD = 0.6        # a late-game world with ~4,000 cells peaks around this
if WORKERS is None:
    WORKERS = max(1, min(cores, int(mem / GB_PER_WORLD)))
if SEEDS_PER_CONFIG is None:
    SEEDS_PER_CONFIG = max(16, WORKERS // len(CONFIGS))
print(f"{cores} usable cores, {mem:.0f} GB free -> {WORKERS} worlds at a time; "
      f"{SEEDS_PER_CONFIG} seeds x {len(CONFIGS)} settings = {SEEDS_PER_CONFIG * len(CONFIGS)} worlds")

def node_version(exe):
    try:
        v = subprocess.run([exe, "--version"], capture_output=True, text=True, timeout=30).stdout.strip()
        return int(v.lstrip("v").split(".")[0]), v
    except Exception:
        return 0, None

NODE = shutil.which("node")
major, ver = node_version(NODE) if NODE else (0, None)
if major < 18:
    arch = {"x86_64": "x64", "amd64": "x64", "aarch64": "arm64", "arm64": "arm64"}.get(platform.machine().lower(), "x64")
    osname = {"linux": "linux", "darwin": "darwin"}.get(sys.platform)
    assert osname, "Install Node.js 18+ yourself on this system (automatic download covers Linux and macOS)"
    name = f"node-v20.18.0-{osname}-{arch}"
    dest = os.path.join(WORK, name)
    if not os.path.isdir(dest):
        ext = "tar.xz" if osname == "linux" else "tar.gz"
        url = f"https://nodejs.org/dist/v20.18.0/{name}.{ext}"
        print("Downloading", url)
        tar = os.path.join(WORK, f"{name}.{ext}")
        urllib.request.urlretrieve(url, tar)
        with tarfile.open(tar) as t:
            t.extractall(WORK)
        os.remove(tar)
    NODE = os.path.join(dest, "bin", "node")
    major, ver = node_version(NODE)
print("Node", ver, "at", NODE)
assert major >= 18, "Node.js 18+ is needed (allow internet access so it can be downloaded, or install it)"


In [ ]:
# ---- The simulation (embedded) --------------------------------------------------------------------
FILES = {
 "check-evolution.mjs": "Ly8gIlJlcGxheSB0aGUgdGFwZSBvZiBsaWZlIjogcnVuIG1hbnkgc2VlZHMgaW4gcGFyYWxsZWwgYW5kIHJlcG9ydCB3aGVuIGVhY2ggbWlsZXN0b25lIGhhcHBlbmVkLgovLyBEb3VibGVzIGFzIGEgcmVncmVzc2lvbiBjaGVjayB0aGF0IGV2ZXJ5IGV2b2x1dGlvbmFyeSBzdGFnZSBzdGlsbCBlbWVyZ2VzIGFmdGVyIGEgY2hhbmdlLgovLwovLyAgIG5vZGUgY2hlY2stZXZvbHV0aW9uLm1qcyBbLS1zZWVkcyAxLThdIFstLXRpY2tzIDYwMDAwXSBbLS1qb2JzIDZdIFstLWNmZyAneyJQUkVEX1AiOjAuNH0nXSBbLS1tZCByZXBvcnQubWRdCmltcG9ydCB7IHNwYXduIH0gZnJvbSAnbm9kZTpjaGlsZF9wcm9jZXNzJzsKaW1wb3J0IG9zIGZyb20gJ25vZGU6b3MnOwppbXBvcnQgZnMgZnJvbSAnbm9kZTpmcyc7CmltcG9ydCB7IGZpbGVVUkxUb1BhdGggfSBmcm9tICdub2RlOnVybCc7Cgpjb25zdCBhcmdzID0gT2JqZWN0LmZyb21FbnRyaWVzKHByb2Nlc3MuYXJndi5zbGljZSgyKS5yZWR1Y2UoKGEsIHYsIGksIGFycikgPT4gKHYuc3RhcnRzV2l0aCgnLS0nKSA/IFsuLi5hLCBbdi5zbGljZSgyKSwgYXJyW2kgKyAxXV1dIDogYSksIFtdKSk7CgovLyBTdGFnZXMgd2UgZXhwZWN0LCBpbiByb3VnaCBoaXN0b3JpY2FsIG9yZGVyLCB3aXRoIHRoZSBzaGFyZSBvZiBzZWVkcyB0aGF0IHNob3VsZCByZWFjaCBlYWNoLgpjb25zdCBTVEFHRVMgPSBbCiAgWydyZXBsaWNhdG9yJywgJ1JlcGxpY2F0b3InLCAxLjBdLAogIFsncHJvdG9jZWxsJywgJ1Byb3RvY2VsbCcsIDEuMF0sCiAgWydjZWxsJywgJ0xpdmluZyBjZWxsJywgMS4wXSwKICBbJ3RyYW5zbGF0aW9uJywgJ1Byb3RlaW5zIC8gZ2VuZXRpYyBjb2RlJywgMC45XSwKICBbJ2RuYScsICdETkEgZ2Vub21lJywgMC44XSwKICBbJ3Bob3RvJywgJ1Bob3Rvc3ludGhlc2lzJywgMC44XSwKICBbJ3ZpcnVzJywgJ1ZpcnVzZXMnLCAwLjhdLAogIFsnZ29lJywgJ0dyZWF0IE94aWRhdGlvbicsIDAuNl0sCiAgWydldWthcnlvdGUnLCAnRXVrYXJ5b3RlJywgMC41XSwKICBbJ3NleCcsICdTZXh1YWwgcmVwcm9kdWN0aW9uJywgMC40XSwKICBbJ211bHRpY2VsbHVsYXInLCAnTXVsdGljZWxsdWxhcicsIDAuNF0sCiAgWydjb21wbGV4JywgJ0NvbXBsZXggb3JnYW5pc20nLCAwLjJdLApdOwoKaWYgKGFyZ3Mud29ya2VyICE9PSB1bmRlZmluZWQpIHsKICAvLyBDaGlsZCBwcm9jZXNzOiBzaW11bGF0ZSBvbmUgc2VlZCBhbmQgcHJpbnQgYSBKU09OIHN1bW1hcnkuCiAgY29uc3QgeyBXb3JsZCB9ID0gYXdhaXQgaW1wb3J0KCcuL2pzL3dvcmxkLmpzJyk7CiAgY29uc3QgeyBDRkcgfSA9IGF3YWl0IGltcG9ydCgnLi9qcy9jb25maWcuanMnKTsKICBpZiAoYXJncy5jZmcpIE9iamVjdC5hc3NpZ24oQ0ZHLCBKU09OLnBhcnNlKGFyZ3MuY2ZnKSk7CiAgY29uc3QgR24gPSBhd2FpdCBpbXBvcnQoJy4vanMvZ2Vub21lLmpzJyk7CiAgY29uc3QgdyA9IG5ldyBXb3JsZCgrYXJncy5zZWVkKTsKICBjb25zdCB0aWNrcyA9ICthcmdzLnRpY2tzOwogIC8vIFdhdGNoICh3aXRob3V0IGNoYW5naW5nIGFueXRoaW5nKSB0aGUgZmlyc3QgY2VsbHMgdG8gY29tZSBhbGl2ZTogdGhlaXIgZ2Vub21lcyBhcmUgYnVpbHQgZnJvbQogIC8vIHRoZSBmb3VuZGluZyByZXBsaWNhdG9yJ3Mgc2VxdWVuY2UsIHNvIGVhcmx5IGNoZW1pc3RyeSBzaGFwZXMgZXZlcnkgbGluZWFnZSBhZnRlciBpdC4KICBjb25zdCBmb3VuZGVycyA9IFtdOwogIGNvbnN0IHNwYXduRnJvbSA9IHcubGlmZS5zcGF3bkZyb21Qcm90b2NlbGwuYmluZCh3LmxpZmUpOwogIHcubGlmZS5zcGF3bkZyb21Qcm90b2NlbGwgPSAodiwgcG9seSkgPT4gewogICAgY29uc3QgYyA9IHNwYXduRnJvbSh2LCBwb2x5KTsKICAgIGlmIChmb3VuZGVycy5sZW5ndGggPCAxMCkgZm91bmRlcnMucHVzaCh7IHQ6IHcudCwgbGVuOiBwb2x5LnNlcS5sZW5ndGgsIGFjdDogK3BvbHkuYWN0LnRvRml4ZWQoMyksIGc6IEFycmF5LmZyb20oYy5nLnNsaWNlKDAsIEduLkRJRkYpLCAoeCkgPT4gK3gudG9GaXhlZCg0KSkgfSk7CiAgICByZXR1cm4gYzsKICB9OwogIGNvbnN0IHQwID0gRGF0ZS5ub3coKTsKICBsZXQgcGVha0NlbGxzID0gMDsKICBjb25zdCBrZXlzID0gU1RBR0VTLm1hcCgoW2tdKSA9PiBrKTsKICB3aGlsZSAody50IDwgdGlja3MpIHsKICAgIHcuc3RlcCgpOwogICAgLy8gRXZlcnkgc3RhZ2UgcmVhY2hlZDogbm8gbmVlZCB0byBrZWVwIHNpbXVsYXRpbmcgdGhpcyB3b3JsZC4KICAgIGlmICh3LnQgJSAxMDAwID09PSAwICYmIGtleXMuZXZlcnkoKGspID0+IHcubXNldC5oYXMoaykpKSBicmVhazsKICAgIGlmICh3LnQgJSAxMDAwID09PSAwKSBwZWFrQ2VsbHMgPSBNYXRoLm1heChwZWFrQ2VsbHMsIHcubGlmZS5jZWxscy5sZW5ndGgpOwogICAgaWYgKHcudCAlIDIwMDAgPT09IDApIHByb2Nlc3Muc3RkZXJyLndyaXRlKGBQUk9HUkVTUyAke3cudH1cbmApOwogIH0KICBjb25zdCBzdCA9IHcubGlmZS5zdGF0czsKICBjb25zdCBtcyA9IE9iamVjdC5mcm9tRW50cmllcyh3Lm1pbGVzdG9uZXMubWFwKChtKSA9PiBbbS5rZXksIG0udF0pKTsKICBjb25zb2xlLmxvZyhKU09OLnN0cmluZ2lmeSh7CiAgICBzZWVkOiArYXJncy5zZWVkLCBtcywgc2VjczogKERhdGUubm93KCkgLSB0MCkgLyAxMDAwLCB0aWNrczogdy50LCBvcmlnaW5zOiB3LmxpZmUub3JpZ2lucywgZm91bmRlcnMsCiAgICBnZW5lTmFtZXM6IGFyZ3MuZ2VuZXMgPyBHbi5HRU5FX05BTUVTLnNsaWNlKDAsIEduLkRJRkYpIDogdW5kZWZpbmVkLAogICAgZmluYWw6IHsgY2VsbHM6IHcubGlmZS5jZWxscy5sZW5ndGgsIHBlYWtDZWxscywgc3BlY2llczogc3Quc3BlY2llc0FsaXZlLCBldWs6IHN0LmV1aywgbXVsdGk6IHN0Lm11bHRpY2VsbHVsYXIsIGxhcmdlc3Q6IHN0Lmxhcmdlc3QsIHZpcnVzZXM6IHcudmlydXNlcyA/IHcudmlydXNlcy5saXN0Lmxlbmd0aCA6IDAsIG8yOiArdy5vMi50b0ZpeGVkKDIpLCBlcmE6IHcuZXJhKCkubmFtZSB9LAogIH0pKTsKICBwcm9jZXNzLmV4aXQoMCk7Cn0KCmNvbnN0IFthLCBiXSA9IChhcmdzLnNlZWRzIHx8ICcxLTgnKS5zcGxpdCgnLScpLm1hcChOdW1iZXIpOwpjb25zdCBzZWVkcyA9IFtdOwpmb3IgKGxldCBzID0gYTsgcyA8PSAoYiB8fCBhKTsgcysrKSBzZWVkcy5wdXNoKHMpOwpjb25zdCB0aWNrcyA9ICsoYXJncy50aWNrcyB8fCA2MDAwMCk7CmNvbnN0IGpvYnMgPSArKGFyZ3Muam9icyB8fCBvcy5jcHVzKCkubGVuZ3RoKTsgLy8gZWFjaCB3b3JsZCBpcyBvbmUgYnVzeSBjb3JlOyB1c2UgdGhlbSBhbGwKY29uc3Qgc2VsZiA9IGZpbGVVUkxUb1BhdGgoaW1wb3J0Lm1ldGEudXJsKTsKCmNvbnNvbGUubG9nKGBSZXBsYXlpbmcgJHtzZWVkcy5sZW5ndGh9IHdvcmxkcyDDlyAke3RpY2tzLnRvTG9jYWxlU3RyaW5nKCl9IHRpY2tzIG9uICR7am9ic30gY29yZXPigKZgKTsKY29uc3QgcmVzdWx0cyA9IFtdOwpjb25zdCBwcm9ncmVzcyA9IG5ldyBNYXAoKTsKbGV0IG5leHQgPSAwOwpjb25zdCB0U3RhcnQgPSBEYXRlLm5vdygpOwpjb25zdCB0aWNrZXIgPSBzZXRJbnRlcnZhbCgoKSA9PiB7CiAgY29uc3QgZG9uZSA9IHJlc3VsdHMubGVuZ3RoLCB0b3RhbCA9IHNlZWRzLmxlbmd0aDsKICBjb25zdCB0aWNrc0RvbmUgPSBbLi4ucHJvZ3Jlc3MudmFsdWVzKCldLnJlZHVjZSgoeCwgeSkgPT4geCArIHksIDApICsgMDsKICBjb25zdCBmcmFjID0gTWF0aC5taW4oMSwgdGlja3NEb25lIC8gKHRvdGFsICogdGlja3MpKTsKICBjb25zdCBlbCA9IChEYXRlLm5vdygpIC0gdFN0YXJ0KSAvIDEwMDA7CiAgY29uc3QgZXRhID0gZnJhYyA+IDAuMDIgPyBlbCAvIGZyYWMgLSBlbCA6IE5hTjsKICBjb25zb2xlLmxvZyhgICDigKYgJHsoZnJhYyAqIDEwMCkudG9GaXhlZCgwKX0lIG92ZXJhbGwgKCR7ZG9uZX0vJHt0b3RhbH0gd29ybGRzIGZpbmlzaGVkLCAke2VsLnRvRml4ZWQoMCl9cyBlbGFwc2VkJHtpc05hTihldGEpID8gJycgOiBgLCB+JHtNYXRoLnJvdW5kKGV0YSl9cyBsZWZ0YH0pYCk7Cn0sIDMwMDAwKTsKYXdhaXQgbmV3IFByb21pc2UoKGRvbmUpID0+IHsKICBsZXQgcnVubmluZyA9IDA7CiAgY29uc3QgbGF1bmNoID0gKCkgPT4gewogICAgd2hpbGUgKHJ1bm5pbmcgPCBqb2JzICYmIG5leHQgPCBzZWVkcy5sZW5ndGgpIHsKICAgICAgY29uc3Qgc2VlZCA9IHNlZWRzW25leHQrK107CiAgICAgIHJ1bm5pbmcrKzsKICAgICAgY29uc3QgY3AgPSBzcGF3bihwcm9jZXNzLmV4ZWNQYXRoLCBbc2VsZiwgJy0td29ya2VyJywgJzEnLCAnLS1zZWVkJywgU3RyaW5nKHNlZWQpLCAnLS10aWNrcycsIFN0cmluZyh0aWNrcyksIC4uLihhcmdzLmNmZyA/IFsnLS1jZmcnLCBhcmdzLmNmZ10gOiBbXSldKTsKICAgICAgbGV0IG91dCA9ICcnOwogICAgICBjcC5zdGRvdXQub24oJ2RhdGEnLCAoZCkgPT4gKG91dCArPSBkKSk7CiAgICAgIGNwLnN0ZGVyci5vbignZGF0YScsIChkKSA9PiB7CiAgICAgICAgZm9yIChjb25zdCBsaW5lIG9mIFN0cmluZyhkKS5zcGxpdCgnXG4nKSkgewogICAgICAgICAgY29uc3QgbSA9IC9eUFJPR1JFU1MgKFxkKykvLmV4ZWMobGluZSk7CiAgICAgICAgICBpZiAobSkgcHJvZ3Jlc3Muc2V0KHNlZWQsICttWzFdKTsgZWxzZSBpZiAobGluZS50cmltKCkpIHByb2Nlc3Muc3RkZXJyLndyaXRlKGxpbmUgKyAnXG4nKTsKICAgICAgICB9CiAgICAgIH0pOwogICAgICBjcC5vbignY2xvc2UnLCAoKSA9PiB7CiAgICAgICAgcnVubmluZy0tOwogICAgICAgIHByb2dyZXNzLnNldChzZWVkLCB0aWNrcyk7CiAgICAgICAgdHJ5IHsgY29uc3QgciA9IEpTT04ucGFyc2Uob3V0LnRyaW0oKS5zcGxpdCgnXG4nKS5wb3AoKSk7IHJlc3VsdHMucHVzaChyKTsgcHJvY2Vzcy5zdGRvdXQud3JpdGUoYCAgc2VlZCAke3NlZWR9IGRvbmUgKCR7ci5zZWNzLnRvRml4ZWQoMCl9cylcbmApOyB9CiAgICAgICAgY2F0Y2ggeyByZXN1bHRzLnB1c2goeyBzZWVkLCBlcnJvcjogdHJ1ZSwgbXM6IHt9LCBmaW5hbDoge30gfSk7IGNvbnNvbGUubG9nKGAgIHNlZWQgJHtzZWVkfSBGQUlMRURgKTsgfQogICAgICAgIGlmIChuZXh0ID49IHNlZWRzLmxlbmd0aCAmJiBydW5uaW5nID09PSAwKSBkb25lKCk7IGVsc2UgbGF1bmNoKCk7CiAgICAgIH0pOwogICAgfQogIH07CiAgbGF1bmNoKCk7Cn0pOwpjbGVhckludGVydmFsKHRpY2tlcik7CnJlc3VsdHMuc29ydCgoeCwgeSkgPT4geC5zZWVkIC0geS5zZWVkKTsKCi8vIFRhYmxlOiBvbmUgcm93IHBlciBzZWVkLCBvbmUgY29sdW1uIHBlciBzdGFnZSAodGljayBpbiB0aG91c2FuZHMpLgpjb25zdCBmbXQgPSAodCkgPT4gKHQgPT09IHVuZGVmaW5lZCA/ICcgICDigJMgICcgOiBgJHsodCAvIDEwMDApLnRvRml4ZWQoMSkucGFkU3RhcnQoNSl9a2ApOwpjb25zdCBoZWFkID0gJ3NlZWQgICcgKyBTVEFHRVMubWFwKChbLCBuXSkgPT4gbi5zbGljZSgwLCA2KS5wYWRTdGFydCg3KSkuam9pbignJykgKyAnICAgY2VsbHMgIHNwICBldWsgdmlydSAgZXJhJzsKY29uc3QgbGluZXMgPSBbaGVhZF07CmZvciAoY29uc3QgciBvZiByZXN1bHRzKSB7CiAgbGluZXMucHVzaChTdHJpbmcoci5zZWVkKS5wYWRTdGFydCg0KSArICcgICcgKyBTVEFHRVMubWFwKChba10pID0+ICcgJyArIGZtdChyLm1zW2tdKSkuam9pbignJykgKwogICAgYCAgJHtTdHJpbmcoci5maW5hbC5jZWxscyA/PyAnJykucGFkU3RhcnQoNil9ICR7U3RyaW5nKHIuZmluYWwuc3BlY2llcyA/PyAnJykucGFkU3RhcnQoMyl9ICR7U3RyaW5nKHIuZmluYWwuZXVrID8/ICcnKS5wYWRTdGFydCg0KX0gJHtTdHJpbmcoci5maW5hbC52aXJ1c2VzID8/ICcnKS5wYWRTdGFydCg0KX0gICR7ci5maW5hbC5lcmEgPz8gJyd9YCk7Cn0KY29uc29sZS5sb2coJ1xuJyArIGxpbmVzLmpvaW4oJ1xuJykpOwoKY29uc29sZS5sb2coJ1xuSG93IG9mdGVuIGVhY2ggc3RhZ2UgaGFwcGVuZWQgKGFuZCBpdHMgbWVkaWFuIHRpbWUpOicpOwpsZXQgb2sgPSB0cnVlOwpjb25zdCBzdW1tYXJ5ID0gW107CmZvciAoY29uc3QgW2ssIG5hbWUsIHdhbnRdIG9mIFNUQUdFUykgewogIGNvbnN0IHRzID0gcmVzdWx0cy5tYXAoKHIpID0+IHIubXNba10pLmZpbHRlcigodCkgPT4gdCAhPT0gdW5kZWZpbmVkKS5zb3J0KCh4LCB5KSA9PiB4IC0geSk7CiAgY29uc3Qgc2hhcmUgPSB0cy5sZW5ndGggLyByZXN1bHRzLmxlbmd0aDsKICBjb25zdCBtZWQgPSB0cy5sZW5ndGggPyB0c1tNYXRoLmZsb29yKHRzLmxlbmd0aCAvIDIpXSA6IHVuZGVmaW5lZDsKICBjb25zdCBwYXNzID0gc2hhcmUgPj0gd2FudDsKICBpZiAoIXBhc3MpIG9rID0gZmFsc2U7CiAgY29uc3Qgcm93ID0gYCAgJHtwYXNzID8gJ+KckycgOiAn4pyXJ30gJHtuYW1lLnBhZEVuZCgyNCl9ICR7U3RyaW5nKE1hdGgucm91bmQoc2hhcmUgKiAxMDApKS5wYWRTdGFydCgzKX0lICAod2FudCDiiaUgJHtNYXRoLnJvdW5kKHdhbnQgKiAxMDApfSUpICAgbWVkaWFuICR7bWVkID09PSB1bmRlZmluZWQgPyAn4oCUJyA6IG1lZC50b0xvY2FsZVN0cmluZygpICsgJyB0aWNrcyd9YDsKICBjb25zb2xlLmxvZyhyb3cpOwogIHN1bW1hcnkucHVzaCh7IG5hbWUsIHNoYXJlLCBtZWQsIHdhbnQsIHBhc3MgfSk7Cn0KY29uc29sZS5sb2cob2sgPyAnXG5BbGwgc3RhZ2VzIGVtZXJnZSBhcyBleHBlY3RlZC4nIDogJ1xuU29tZSBzdGFnZXMgZW1lcmdlIGxlc3Mgb2Z0ZW4gdGhhbiBleHBlY3RlZCAoc2VlIOKclykuJyk7CgppZiAoYXJncy5tZCkgewogIGNvbnN0IG1kID0gW2AjIFJlcGxheWluZyB0aGUgdGFwZSBvZiBsaWZlXG5gLCBgJHtyZXN1bHRzLmxlbmd0aH0gd29ybGRzLCAke3RpY2tzLnRvTG9jYWxlU3RyaW5nKCl9IHRpY2tzIGVhY2guXG5gLAogICAgJ3wgU3RhZ2UgfCBXb3JsZHMgcmVhY2hpbmcgaXQgfCBNZWRpYW4gdGljayB8JywgJ3wgLS0tIHwgLS0tIHwgLS0tIHwnLAogICAgLi4uc3VtbWFyeS5tYXAoKHMpID0+IGB8ICR7cy5uYW1lfSB8ICR7TWF0aC5yb3VuZChzLnNoYXJlICogMTAwKX0lIHwgJHtzLm1lZCA9PT0gdW5kZWZpbmVkID8gJ+KAlCcgOiBzLm1lZC50b0xvY2FsZVN0cmluZygpfSB8YCksCiAgICAnJywgJ2BgYCcsIC4uLmxpbmVzLCAnYGBgJywgJyddOwogIGZzLndyaXRlRmlsZVN5bmMoYXJncy5tZCwgbWQuam9pbignXG4nKSk7CiAgY29uc29sZS5sb2coYFdyb3RlICR7YXJncy5tZH1gKTsKfQpwcm9jZXNzLmV4aXQob2sgPyAwIDogMSk7Cg==",
 "js/chemistry.js": "...",
 "js/config.js": "Ly8gQWxsIHR1bmFibGUgY29uc3RhbnRzIG9mIHRoZSBzaW11bGF0aW9uIGluIG9uZSBwbGFjZS4KZXhwb3J0IGNvbnN0IENGRyA9IHsKICAvLyBXb3JsZCB1bml0cyBhcmUgMC41IMK1bTogdGhlIGhhYml0YXQgaXMgMS42IMOXIDAuOSBtbS4KICBXOiAzMjAwLAogIEg6IDE4MDAsCgogIC8vIC0tLS0gRW52aXJvbm1lbnQgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogIEZJRUxEX0NFTEw6IDIwLAogIFZFTlRTOiA3LAogIFZFTlRfQ0hFTTogNC4wLCAgICAgICAgICAvLyByZWR1Y2VkIGNoZW1pY2FscyAoSDIsIEgyUykgZW1pdHRlZCBwZXIgdmVudCBwZXIgdGljawogIFZFTlRfR0xPVzogMC4zLCAgICAgICAgIC8vIGZhaW50IHRoZXJtYWwgbGlnaHQgbmVhciB2ZW50cyAoYW5veHlnZW5pYyBwaG90b3N5bnRoZXNpcyBzdGVwcGluZyBzdG9uZSkKICBDSEVNX0RJRkY6IDAuMjUsIENIRU1fREVDQVk6IDAuMDAxNSwgQ0hFTV9IQUxGOiA0LAogIE9SR19ESUZGOiAwLjE1LCBPUkdfREVDQVk6IDAuMDAwNiwKICBPWFlfRElGRjogMC4zLCBPWFlfREVDQVk6IDAuMDAxNSwgT1hZX1JFQUNUOiAwLjAxLAogIE5VVF9JTklUOiAwLjMsICAgICAgICAgICAvLyBjb25zZXJ2ZWQgbGltaXRpbmcgbnV0cmllbnQgKHRoaW5rIHBob3NwaGF0ZSkgcGVyIGdyaWQgY2VsbAogIE5VVF9ESUZGOiAwLjMsIE5VVF9CVVJJQUw6IDAuMDAwNCwKICBWRU5UX05VVDogMC4xLCAgICAgICAgICAvLyBtaW5lcmFscyBmcm9tIHZlbnRzCiAgV0VBVEhFUklORzogMC4xNiwgICAgICAgICAvLyBtaW5lcmFscyB3YXNoZWQgaW4gYXQgdGhlIHN1cmZhY2UgKHJpdmVycywgZHVzdCkKICBPMl9SRUY6IDEuNSwgICAgICAgICAgICAgICAvLyBveHlnZW4gbGV2ZWwgY29uc2lkZXJlZCAiZnVsbHkgb3h5Z2VuYXRlZCIKICBEQVlfTEVOR1RIOiA0MDAwLAogIExJR0hUX0RFUFRIOiAxLjYsICAgICAgICAvLyBsaWdodCBhdHRlbnVhdGlvbiB3aXRoIGRlcHRoCiAgQ1VSUkVOVDogMC4xMiwKICBDRUxMX0RSSUZUOiAxLjIsICAgICAgICAgLy8gaG93IHN0cm9uZ2x5IGN1cnJlbnRzIGNhcnJ5IGNlbGxzIChtaWNyb2JlcyBhcmUgbW9zdGx5IHBhc3NpdmUpCiAgUExVTUVfUkVBQ0g6IDAuMzUsICAgICAgIC8vIHZlbnQgcGx1bWVzIHJpc2UgdGhpcyBmcmFjdGlvbiBvZiB0aGUgb2NlYW4gZGVwdGgKCiAgLy8gLS0tLSBQcmViaW90aWMgY2hlbWlzdHJ5IC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgSEFTSF9DRUxMOiAxNiwKICBNQVhfTU9OT01FUlM6IDM1MDAsCiAgVkVOVF9NT05PTUVSX1JBVEU6IDEuMywgIC8vIHBlciB2ZW50IHBlciB0aWNrCiAgU1VSRkFDRV9NT05PTUVSX1JBVEU6IDAuMzUsIC8vIE1pbGxlcuKAk1VyZXkgc3R5bGUgc3ludGhlc2lzIGJ5IGxpZ2h0bmluZy9VViBhdCB0aGUgc3VyZmFjZQogIExJUElEX0ZSQUNUSU9OOiAwLjMsCiAgTU9OT01FUl9ERUNBWTogMC4wMDA0LAogIE1PTk9fRElGRlVTSU9OOiAxLjEsCiAgTUFYX1BPTFlNRVJTOiA4MDAsCiAgTUFYX1BPTFlfTEVOOiA0MCwKICBESU1FUl9TQU1QTEVTOiAyNTAsCiAgRElNRVJfUkFURTogMC4yNSwKICBQT0xZX1JBVEU6IDAuMiwKICBIWURST0xZU0lTOiAwLjAwMDQsCiAgVVZfREFNQUdFOiAxLjUsCiAgUkVQX1JBVEU6IDAuOCwKICBNSU5fQUNUSVZFX0xFTjogNiwKICBNQVhfVkVTSUNMRVM6IDI2MCwKICBWRVNfTlVDTEVBVEU6IDgsCiAgVkVTX1NQTElUOiAzNCwKICBWRVNfUE9PTDogNjAsCiAgVkVTX05UX1JFQUNIOiAxLjUsICAgICAgICAvLyBob3cgZmFyIGJleW9uZCBpdHMgbWVtYnJhbmUgYSBwcm90b2NlbGwgZHJhd3MgaW4gbnVjbGVvdGlkZXMKICBWRVNfTUFYX1BPTFk6IDE0LAogIFZFU19MSVBJRF9MT1NTOiAwLjAxMiwKICBWRVNfQlVSU1Q6IDAuMDAwMjUsCiAgQ0VMTF9HRU46IDIsICAgICAgICAgICAgIC8vIHByb3RvY2VsbCBnZW5lcmF0aW9ucyBuZWVkZWQgYmVmb3JlIGl0IGNvdW50cyBhcyBhIGxpdmluZyBjZWxsCiAgQ0VMTF9BQ1Q6IDAuMiwgICAgICAgICAgIC8vIG1pbmltdW0gcmlib3p5bWUgYWN0aXZpdHkgb2YgdGhlIHByb3RvY2VsbCdzIGJlc3QgcmVwbGljYXRvcgogIENFTExfQ09QSUVTOiAxLCAgICAgICAgICAvLyBjb21wbGV0ZWQgc2VsZi1jb3BpZXMgaW4gdGhlIHByb3RvY2VsbCBsaW5lYWdlIChoZXJlZGl0eSkKICBNQVhfT1JJR0lOU19QT1A6IDIwMCwgICAgLy8gc3RvcCBuZXcgYWJpb2dlbmVzaXMgb25jZSBsaWZlIGlzIHRoaXMgYWJ1bmRhbnQKCiAgLy8gLS0tLSBMaWZlIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogIE1BWF9DRUxMUzogNDAwMCwKICBMSUZFX0hBU0g6IDI0LAogIE1BWF9SOiAxNCwKICBDQVBfSzogMjAsCiAgQkFTRV9DT1NUOiAwLjAzLAogIE1BQ0hJTkVfQ09TVDogMC4wMDYsCiAgUEhPVE9fSzogMC40NSwKICBDSEVNT19LOiAwLjA4LCBDSEVNT19FRkY6IDIuNSwKICBIRVRfSzogMC4wOCwgSEVUX0VGRjogMi4yLAogIFNQRUNJQUxJWkU6IDEuMTUsICAgICAgICAvLyA+MSBtZWFucyBzcGVjaWFsaXN0cyBiZWF0IGdlbmVyYWxpc3RzCiAgQUVST19CT05VUzogMi4wLAogIEFFUl9DT1NUOiAwLjAwNiwKICBBUk1fQ09TVDogMC4wMTUsCiAgQURIX0NPU1Q6IDAuMDAyLAogIE1PVF9DT1NUOiAwLjAxNSwKICBNT1RfU1BFRUQ6IDAuMzUsCiAgTzJfWUlFTEQ6IDAuMSwKICBUT1g6IDAuMDUsCiAgRElWX0NPU1Q6IDAuMDYsCiAgTUlOX0RJVl9BR0U6IDQwLAogIFBSRURfUDogMC4yNSwKICBFTkRPX1A6IDAuMDUsCiAgSEdUX1A6IDAuMDAwNCwKICBTUEVDSUFURTogMC4wNywKICBEQU1QOiAwLjgyLAogIEJPTkRfQlJFQUs6IDAuMDAwMTUsCiAgU0hBUkVfSzogMC4xLAogIE1BWF9CT05EUzogNSwKICBHRU5FUkFMSVNUX0JJQVM6IDAuMiwKCiAgLy8gTWFqb3IgdHJhbnNpdGlvbnMKICBSSUJPU09NRV9DT1NUOiAwLjAwNCwgICAgLy8gdXBrZWVwIG9mIHByb3RlaW4tc3ludGhlc2lzIG1hY2hpbmVyeSwgcGVyIHVuaXQgdHJhbnNsYXRpb24KICBSTkFfR0VOT01FX0NPU1Q6IDAuMDA0LCAgLy8gcmVwYWlyaW5nIGEgZnJhZ2lsZSBSTkEgZ2Vub21lICjDlzMgaW4gZnVsbCBveHlnZW4pCiAgRE5BX0NPU1Q6IDAuMDAyLCAgICAgICAgIC8vIEROQSBnZW5vbWUgdXBrZWVwCiAgRE5BX01VVDogMC4zNSwgICAgICAgICAgIC8vIG11dGF0aW9uLXJhdGUgbXVsdGlwbGllciBmb3IgcHJvb2ZyZWFkIEROQSBnZW5vbWVzCiAgU0VYX0NPU1Q6IDAuMDgsICAgICAgICAgIC8vIGVuZXJneSBlYWNoIHBhcmVudCBzcGVuZHMgb24gYSBzZXh1YWwgYmlydGggKGZyYWN0aW9uIG9mIGNhcGFjaXR5KQoKICAvLyBWaXJ1c2VzCiAgVklSVVNfT1JJR0lOX1BPUDogNDAwLCAgIC8vIGNlbGxzIG5lZWRlZCBiZWZvcmUgc2VsZmlzaCBlbGVtZW50cyBjYW4gZXNjYXBlIGFzIHZpcnVzZXMKICBWSVJVU19PUklHSU5fUDogMC4wMDA1LAogIFZJUlVTX0RFQ0FZOiAwLjAwMywgICAgICAvLyBwZXIgdGljayAow5c0IGF0IHRoZSBzdXJmYWNlOiBVVikKICBWSVJVU19EOiAxLjUsICAgICAgICAgICAgLy8gQnJvd25pYW4gc3RlcCAodW5pdHMvdGljaykKICBWSVJVU19JTkZFQ1Q6IDAuMzUsCiAgVklSVVNfU1BFQzogMC4xMiwgICAgICAgIC8vIGhvdyBjbG9zZWx5IHRoZSBrZXkgbXVzdCBtYXRjaCB0aGUgcmVjZXB0b3IKICBWSVJVU19MQVRFTkNZOiA2MCwKICBWSVJVU19CVVJTVDogMTAsCiAgVklSVVNfTVVUOiAwLjA0LAogIE1BWF9WSVJVU0VTOiA0MDAwLAogIFRSQU5TRFVDRTogMC4xLAogIFJFU0lTVF9DT1NUOiAwLjAwOCwgICAgLy8gaG93IHN0cm9uZ2x5IGNvbG9ueSBjZWxscyBkZWZhdWx0IHRvIHRoZSB1bmRpZmZlcmVudGlhdGVkIGZhdGUKICBERUFUSF9PUkc6IDAuNiwKICBOVVRfUEVSX0FSRUE6IDIuMCwgICAgICAgLy8gbnV0cmllbnQgbG9ja2VkIGludG8gZWFjaCB1bml0IG9mIGNlbGwgYm9keQogIE5VVF9VUFRBS0U6IDAuMDYsCn07Cg==",
 "js/fields.js": "Ly8gRGlmZnVzaW5nIGNoZW1pY2FsIGNvbmNlbnRyYXRpb24gZmllbGRzIG9uIGEgY29hcnNlIGdyaWQuCmV4cG9ydCBjbGFzcyBGaWVsZHMgewogIGNvbnN0cnVjdG9yKFcsIEgsIGNzKSB7CiAgICB0aGlzLmNzID0gY3M7CiAgICB0aGlzLmd3ID0gTWF0aC5jZWlsKFcgLyBjcyk7CiAgICB0aGlzLmdoID0gTWF0aC5jZWlsKEggLyBjcyk7CiAgICB0aGlzLm4gPSB0aGlzLmd3ICogdGhpcy5naDsKICAgIHRoaXMuY2hlbSA9IG5ldyBGbG9hdDMyQXJyYXkodGhpcy5uKTsgICAgIC8vIHJlZHVjZWQgY2hlbWljYWxzIGZyb20gdmVudHMKICAgIHRoaXMub3JnYW5pYyA9IG5ldyBGbG9hdDMyQXJyYXkodGhpcy5uKTsgIC8vIGRpc3NvbHZlZCBvcmdhbmljIG1hdHRlcgogICAgdGhpcy5veHlnZW4gPSBuZXcgRmxvYXQzMkFycmF5KHRoaXMubik7ICAgLy8gZnJlZSBPMiBmcm9tIHBob3Rvc3ludGhlc2lzCiAgICB0aGlzLm51dHJpZW50ID0gbmV3IEZsb2F0MzJBcnJheSh0aGlzLm4pOyAvLyBsaW1pdGluZyBtaW5lcmFsIG51dHJpZW50LCByZWN5Y2xlZCB0aHJvdWdoIGJvZGllcwogICAgdGhpcy50bXAgPSBuZXcgRmxvYXQzMkFycmF5KHRoaXMubik7CiAgfQoKICBpZHgoeCwgeSkgewogICAgbGV0IGN4ID0gKHggLyB0aGlzLmNzKSB8IDAsIGN5ID0gKHkgLyB0aGlzLmNzKSB8IDA7CiAgICBpZiAoY3ggPCAwKSBjeCA9IDA7IGVsc2UgaWYgKGN4ID49IHRoaXMuZ3cpIGN4ID0gdGhpcy5ndyAtIDE7CiAgICBpZiAoY3kgPCAwKSBjeSA9IDA7IGVsc2UgaWYgKGN5ID49IHRoaXMuZ2gpIGN5ID0gdGhpcy5naCAtIDE7CiAgICByZXR1cm4gY3kgKiB0aGlzLmd3ICsgY3g7CiAgfQoKICBnZXQoZiwgeCwgeSkgeyByZXR1cm4gZlt0aGlzLmlkeCh4LCB5KV07IH0KICBhZGQoZiwgeCwgeSwgYSkgeyBmW3RoaXMuaWR4KHgsIHkpXSArPSBhOyB9CiAgdGFrZShmLCB4LCB5LCBhKSB7CiAgICBjb25zdCBpID0gdGhpcy5pZHgoeCwgeSk7CiAgICBjb25zdCB2ID0gZltpXTsKICAgIGNvbnN0IHQgPSB2IDwgYSA/IHYgOiBhOwogICAgZltpXSA9IHYgLSB0OwogICAgcmV0dXJuIHQ7CiAgfQoKICAvLyBXcml0ZXMgYSByb3VnaGx5IHVuaXQgZ3JhZGllbnQgdmVjdG9yIGludG8gb3V0OyBtYWduaXR1ZGUgc2hyaW5rcyB3aGVuIHRoZSBmaWVsZCBpcyBmbGF0LgogIGdyYWQoZiwgeCwgeSwgb3V0KSB7CiAgICBjb25zdCBndyA9IHRoaXMuZ3csIGdoID0gdGhpcy5naDsKICAgIGxldCBjeCA9ICh4IC8gdGhpcy5jcykgfCAwLCBjeSA9ICh5IC8gdGhpcy5jcykgfCAwOwogICAgaWYgKGN4IDwgMSkgY3ggPSAxOyBlbHNlIGlmIChjeCA+IGd3IC0gMikgY3ggPSBndyAtIDI7CiAgICBpZiAoY3kgPCAxKSBjeSA9IDE7IGVsc2UgaWYgKGN5ID4gZ2ggLSAyKSBjeSA9IGdoIC0gMjsKICAgIGNvbnN0IGkgPSBjeSAqIGd3ICsgY3g7CiAgICBjb25zdCBneCA9IGZbaSArIDFdIC0gZltpIC0gMV07CiAgICBjb25zdCBneSA9IGZbaSArIGd3XSAtIGZbaSAtIGd3XTsKICAgIGNvbnN0IG0gPSBNYXRoLmh5cG90KGd4LCBneSk7CiAgICBpZiAobSA8IDFlLTYpIHsgb3V0LnggPSAwOyBvdXQueSA9IDA7IHJldHVybiBvdXQ7IH0KICAgIGNvbnN0IHJlbCA9IE1hdGgubWluKDEsIChtIC8gKGZbaV0gKyAwLjUpKSAqIDUpOwogICAgb3V0LnggPSAoZ3ggLyBtKSAqIHJlbDsKICAgIG91dC55ID0gKGd5IC8gbSkgKiByZWw7CiAgICByZXR1cm4gb3V0OwogIH0KCiAgZGlmZnVzZShmLCByYXRlLCBkZWNheSkgewogICAgY29uc3QgZ3cgPSB0aGlzLmd3LCBnaCA9IHRoaXMuZ2gsIHQgPSB0aGlzLnRtcDsKICAgIGNvbnN0IGtlZXAgPSAxIC0gZGVjYXk7CiAgICBmb3IgKGxldCB5ID0gMDsgeSA8IGdoOyB5KyspIHsKICAgICAgY29uc3Qgcm93ID0geSAqIGd3OwogICAgICBmb3IgKGxldCB4ID0gMDsgeCA8IGd3OyB4KyspIHsKICAgICAgICBjb25zdCBpID0gcm93ICsgeDsKICAgICAgICBjb25zdCBjID0gZltpXTsKICAgICAgICBjb25zdCBsID0geCA+IDAgPyBmW2kgLSAxXSA6IGM7CiAgICAgICAgY29uc3QgciA9IHggPCBndyAtIDEgPyBmW2kgKyAxXSA6IGM7CiAgICAgICAgY29uc3QgdSA9IHkgPiAwID8gZltpIC0gZ3ddIDogYzsKICAgICAgICBjb25zdCBkID0geSA8IGdoIC0gMSA/IGZbaSArIGd3XSA6IGM7CiAgICAgICAgdFtpXSA9IChjICsgcmF0ZSAqICgobCArIHIgKyB1ICsgZCkgKiAwLjI1IC0gYykpICoga2VlcDsKICAgICAgfQogICAgfQogICAgZi5zZXQodCk7CiAgfQoKICBtZWFuKGYpIHsKICAgIGxldCBzID0gMDsKICAgIGZvciAobGV0IGkgPSAwOyBpIDwgZi5sZW5ndGg7IGkrKykgcyArPSBmW2ldOwogICAgcmV0dXJuIHMgLyBmLmxlbmd0aDsKICB9Cn0K",
 "js/genome.js": "Ly8gRml4ZWQtbG9jdXMgZ2Vub21lLiBUcmFpdCBnZW5lcyBsaXZlIGluIFswLDFdOyBiZWhhdmlvdXIvcmVndWxhdG9yeSB3ZWlnaHRzIGluIFstMSwxXS4KaW1wb3J0IHsgUk5HLCBoYXNoU2VxIH0gZnJvbSAnLi9ybmcuanMnOwoKZXhwb3J0IGNvbnN0IFNJWkUgPSAwLCBQSE9UTyA9IDEsIENIRU1PID0gMiwgSEVURVJPID0gMywgUFJFRCA9IDQsIE1PVElMID0gNSwgQUVST0JJQyA9IDYsCiAgRElWVEggPSA3LCBBREhFUyA9IDgsIFNIQVJFID0gOSwgQVJNT1IgPSAxMCwgTVVUUiA9IDExLCBIVUUgPSAxMiwgTElGRVNQQU4gPSAxMywgQ09MTUFYID0gMTQ7Ci8vIE1ham9yLXRyYW5zaXRpb24gYW5kIGFybXMtcmFjZSBnZW5lcy4KZXhwb3J0IGNvbnN0IFRSQU5TTCA9IDE1OyAgIC8vIHByb3RlaW4gdHJhbnNsYXRpb246IHJpYm9zb21lcyArIGdlbmV0aWMgY29kZSByZXBsYWNlIHJpYm96eW1lIG1ldGFib2xpc20KZXhwb3J0IGNvbnN0IEROQUcgPSAxNjsgICAgIC8vIEROQSBnZW5vbWUgKD4gMC41KTogc3RhYmxlIHN0b3JhZ2UgaW5zdGVhZCBvZiBSTkEKZXhwb3J0IGNvbnN0IFNFWCA9IDE3OyAgICAgIC8vIHRlbmRlbmN5IHRvIHJlcHJvZHVjZSBzZXh1YWxseSAoZXVrYXJ5b3RlcyBvbmx5KQpleHBvcnQgY29uc3QgUkVDMSA9IDE4LCBSRUMyID0gMTk7IC8vIHNoYXBlIG9mIHN1cmZhY2UgcmVjZXB0b3JzOiB3aGF0IHZpcnVzZXMgbGF0Y2ggb250bwpleHBvcnQgY29uc3QgUkVTSVNUID0gMjA7ICAgLy8gQ1JJU1BSLWxpa2UgYW50aXZpcmFsIGRlZmVuY2UKZXhwb3J0IGNvbnN0IFdfTElHSFQgPSAyMSwgV19DSEVNID0gMjIsIFdfT1JHID0gMjMsIFdfUFJFWSA9IDI0LCBXX0ZMRUUgPSAyNSwgV19PWFkgPSAyNjsKZXhwb3J0IGNvbnN0IEZJUlNUX1NJR05FRCA9IFdfTElHSFQ7Ci8vIEdlbmUtcmVndWxhdG9yeSB3ZWlnaHRzIGRlY2lkaW5nIGNlbGwgZmF0ZSBpbnNpZGUgYSBjb2xvbnk6IGZvciBlYWNoIGNlbGwgdHlwZSwKLy8gYSB3ZWlnaHRlZCBzdW0gb2YgW2JpYXMsICNuZWlnaGJvdXJzLCBkZXB0aCBmcm9tIGZvdW5kZXIsIGNvbG9ueSBzaXplLCBsaWdodF0uCmV4cG9ydCBjb25zdCBESUZGID0gMjcsIE5UWVBFUyA9IDYsIE5ESUZGX0lOID0gNTsKZXhwb3J0IGNvbnN0IE5HID0gRElGRiArIE5UWVBFUyAqIE5ESUZGX0lOOwoKZXhwb3J0IGNvbnN0IEdFTkVfTkFNRVMgPSBbCiAgJ1NpemUnLCAnUGhvdG9zeW50aGVzaXMnLCAnQ2hlbW9zeW50aGVzaXMnLCAnSGV0ZXJvdHJvcGh5JywgJ1ByZWRhdGlvbicsICdNb3RpbGl0eScsICdBZXJvYmljIHJlc3AuJywKICAnRGl2aXNpb24gdGhyZXNob2xkJywgJ0FkaGVzaW9uJywgJ0VuZXJneSBzaGFyaW5nJywgJ0FybW9yJywgJ011dGF0aW9uIHJhdGUnLCAnUGlnbWVudCBodWUnLCAnTGlmZXNwYW4nLCAnQ29sb255IHNpemUnLAogICdQcm90ZWluIHRyYW5zbGF0aW9uJywgJ0ROQSBnZW5vbWUnLCAnU2V4dWFsIHJlcHJvZHVjdGlvbicsICdSZWNlcHRvciBzaGFwZSBBJywgJ1JlY2VwdG9yIHNoYXBlIEInLCAnVmlyYWwgZGVmZW5jZScsCiAgJ1NlZWsgbGlnaHQnLCAnU2VlayBjaGVtaWNhbHMnLCAnU2VlayBvcmdhbmljcycsICdDaGFzZSBwcmV5JywgJ0ZsZWUgcHJlZGF0b3JzJywgJ1NlZWsgb3h5Z2VuJywKXTsKCi8vIFRoZSBmaXJzdCBjZWxsJ3MgZ2Vub21lIGlzIGRlcml2ZWQgZnJvbSB0aGUgc2VxdWVuY2Ugb2YgdGhlIHJlcGxpY2F0b3IgdGhhdCBmb3VuZGVkIGl0LgovLyBJdCBpcyBhbiBSTkEtd29ybGQgY2VsbDogcmlib3p5bWVzIGRvIHRoZSBjaGVtaXN0cnkgYW5kIHRoZSBnZW5vbWUgaXMgUk5BLgpleHBvcnQgZnVuY3Rpb24gZnJvbVNlcXVlbmNlKHNlcSkgewogIGNvbnN0IHIgPSBuZXcgUk5HKGhhc2hTZXEoc2VxKSk7CiAgY29uc3QgZyA9IG5ldyBGbG9hdDMyQXJyYXkoTkcpOwogIGdbU0laRV0gPSAwLjI1ICsgci5uZXh0KCkgKiAwLjE1OwogIGdbUEhPVE9dID0gci5uZXh0KCkgKiAwLjA0OwogIGdbQ0hFTU9dID0gMC42ICsgci5uZXh0KCkgKiAwLjI7CiAgZ1tIRVRFUk9dID0gMC4xNSArIHIubmV4dCgpICogMC4yOwogIGdbUFJFRF0gPSByLm5leHQoKSAqIDAuMDM7CiAgZ1tNT1RJTF0gPSAwLjEgKyByLm5leHQoKSAqIDAuMjsKICBnW0FFUk9CSUNdID0gci5uZXh0KCkgKiAwLjAzOwogIGdbRElWVEhdID0gMC40ICsgci5uZXh0KCkgKiAwLjM7CiAgZ1tBREhFU10gPSByLm5leHQoKSAqIDAuMDY7CiAgZ1tTSEFSRV0gPSByLm5leHQoKSAqIDAuMjsKICBnW0FSTU9SXSA9IHIubmV4dCgpICogMC4wNTsKICBnW01VVFJdID0gMC4zICsgci5uZXh0KCkgKiAwLjI7CiAgZ1tIVUVdID0gci5uZXh0KCk7CiAgZ1tMSUZFU1BBTl0gPSAwLjMgKyByLm5leHQoKSAqIDAuMzsKICBnW0NPTE1BWF0gPSByLm5leHQoKSAqIDAuMTsKICBnW1RSQU5TTF0gPSAwLjE1ICsgci5uZXh0KCkgKiAwLjE1OwogIGdbRE5BR10gPSByLm5leHQoKSAqIDAuMTsKICBnW1NFWF0gPSAwLjA1ICsgci5uZXh0KCkgKiAwLjE7IC8vIG9ubHkgZXhwcmVzc2VkIG9uY2UgYSBsaW5lYWdlIGJlY29tZXMgZXVrYXJ5b3RpYwogIGdbUkVDMV0gPSByLm5leHQoKTsKICBnW1JFQzJdID0gci5uZXh0KCk7CiAgZ1tSRVNJU1RdID0gci5uZXh0KCkgKiAwLjA1OwogIGdbV19MSUdIVF0gPSAoci5uZXh0KCkgLSAwLjUpICogMC40OwogIGdbV19DSEVNXSA9IDAuNCArIHIubmV4dCgpICogMC4zOwogIGdbV19PUkddID0gci5uZXh0KCkgKiAwLjI7CiAgZ1tXX1BSRVldID0gci5uZXh0KCkgKiAwLjM7CiAgZ1tXX0ZMRUVdID0gci5uZXh0KCkgKiAwLjM7CiAgZ1tXX09YWV0gPSAtci5uZXh0KCkgKiAwLjI7CiAgZm9yIChsZXQgaSA9IERJRkY7IGkgPCBORzsgaSsrKSBnW2ldID0gKHIubmV4dCgpIC0gMC41KSAqIDAuMzsKICByZXR1cm4gZzsKfQoKLy8gcmF0ZU11bCA8IDEgZm9yIEROQSBnZW5vbWVzIChwcm9vZnJlYWRpbmcgYW5kIHJlcGFpcikuCmV4cG9ydCBmdW5jdGlvbiBtdXRhdGUocCwgcm5nLCByYXRlTXVsID0gMSkgewogIGNvbnN0IGcgPSBuZXcgRmxvYXQzMkFycmF5KHApOwogIGNvbnN0IHJhdGUgPSAoMC4wMSArIHBbTVVUUl0gKiAwLjA1KSAqIHJhdGVNdWw7IC8vIHRoZSBtdXRhdGlvbiByYXRlIGl0c2VsZiBldm9sdmVzCiAgZm9yIChsZXQgaSA9IDA7IGkgPCBORzsgaSsrKSB7CiAgICBpZiAoaSA9PT0gSFVFIHx8IHJuZy5uZXh0KCkgPj0gcmF0ZSkgY29udGludWU7CiAgICBpZiAocm5nLm5leHQoKSA8IDAuMDQpIGdbaV0gPSBpID49IEZJUlNUX1NJR05FRCA/IHJuZy5yYW5nZSgtMSwgMSkgOiBybmcubmV4dCgpOyAvLyBsYXJnZS1lZmZlY3QgbXV0YXRpb24KICAgIGVsc2UgZ1tpXSArPSBybmcuZ2F1c3MoKSAqIDAuMTsKICAgIGNvbnN0IGxvID0gaSA+PSBGSVJTVF9TSUdORUQgPyAtMSA6IDA7CiAgICBpZiAoZ1tpXSA8IGxvKSBnW2ldID0gbG87IGVsc2UgaWYgKGdbaV0gPiAxKSBnW2ldID0gMTsKICB9CiAgLy8gTmV1dHJhbCBwaWdtZW50IGRyaWZ0cyBldmVyeSBnZW5lcmF0aW9uOiBhIHZpc2libGUgbW9sZWN1bGFyIGNsb2NrLgogIGdbSFVFXSA9IChnW0hVRV0gKyBybmcuZ2F1c3MoKSAqIDAuMDA0ICsgMSkgJSAxOwogIHJldHVybiBnOwp9CgovLyBTZXh1YWwgcmVwcm9kdWN0aW9uOiBlYWNoIGdlbmUgY29tZXMgZnJvbSBvbmUgcGFyZW50IG9yIHRoZSBvdGhlciAoZnJlZSByZWNvbWJpbmF0aW9uKS4KZXhwb3J0IGZ1bmN0aW9uIHJlY29tYmluZShhLCBiLCBybmcpIHsKICBjb25zdCBnID0gbmV3IEZsb2F0MzJBcnJheShhLmxlbmd0aCk7CiAgZm9yIChsZXQgaSA9IDA7IGkgPCBhLmxlbmd0aDsgaSsrKSBnW2ldID0gcm5nLm5leHQoKSA8IDAuNSA/IGFbaV0gOiBiW2ldOwogIHJldHVybiBnOwp9CgpleHBvcnQgZnVuY3Rpb24gZGlzdGFuY2UoYSwgYikgewogIGxldCBzID0gMDsKICBmb3IgKGxldCBpID0gMDsgaSA8IE5HOyBpKyspIHsKICAgIGlmIChpID09PSBIVUUpIGNvbnRpbnVlOwogICAgY29uc3QgZCA9IE1hdGguYWJzKGFbaV0gLSBiW2ldKTsKICAgIHMgKz0gaSA+PSBGSVJTVF9TSUdORUQgPyBkICogMC41IDogZDsKICB9CiAgcmV0dXJuIHMgLyAoTkcgLSAxKTsKfQo=",
 "js/life.js": "...",
 "js/rng.js": "Ly8gU2VlZGVkIFBSTkcgKG11bGJlcnJ5MzIpIHNvIGV2ZXJ5IHdvcmxkIGlzIHJlcHJvZHVjaWJsZSBmcm9tIGl0cyBzZWVkLgpleHBvcnQgY2xhc3MgUk5HIHsKICBjb25zdHJ1Y3RvcihzZWVkKSB7CiAgICB0aGlzLnMgPSAoc2VlZCA+Pj4gMCkgfHwgMTsKICAgIHRoaXMuX3NwYXJlID0gbnVsbDsKICB9CiAgbmV4dCgpIHsKICAgIGxldCB0ID0gKHRoaXMucyA9ICh0aGlzLnMgKyAweDZkMmI3OWY1KSA+Pj4gMCk7CiAgICB0ID0gTWF0aC5pbXVsKHQgXiAodCA+Pj4gMTUpLCB0IHwgMSk7CiAgICB0IF49IHQgKyBNYXRoLmltdWwodCBeICh0ID4+PiA3KSwgdCB8IDYxKTsKICAgIHJldHVybiAoKHQgXiAodCA+Pj4gMTQpKSA+Pj4gMCkgLyA0Mjk0OTY3Mjk2OwogIH0KICByYW5nZShhLCBiKSB7IHJldHVybiBhICsgKGIgLSBhKSAqIHRoaXMubmV4dCgpOyB9CiAgaW50KG4pIHsgcmV0dXJuIE1hdGguZmxvb3IodGhpcy5uZXh0KCkgKiBuKTsgfQogIGNoYW5jZShwKSB7IHJldHVybiB0aGlzLm5leHQoKSA8IHA7IH0KICBnYXVzcygpIHsKICAgIGlmICh0aGlzLl9zcGFyZSAhPT0gbnVsbCkgeyBjb25zdCBzID0gdGhpcy5fc3BhcmU7IHRoaXMuX3NwYXJlID0gbnVsbDsgcmV0dXJuIHM7IH0KICAgIGxldCB1ID0gMCwgdiA9IDA7CiAgICB3aGlsZSAodSA9PT0gMCkgdSA9IHRoaXMubmV4dCgpOwogICAgdiA9IHRoaXMubmV4dCgpOwogICAgY29uc3QgbSA9IE1hdGguc3FydCgtMiAqIE1hdGgubG9nKHUpKTsKICAgIHRoaXMuX3NwYXJlID0gbSAqIE1hdGguc2luKDYuMjgzMTg1MzA3ICogdik7CiAgICByZXR1cm4gbSAqIE1hdGguY29zKDYuMjgzMTg1MzA3ICogdik7CiAgfQp9CgpleHBvcnQgZnVuY3Rpb24gaGFzaFNlcShzZXEpIHsKICBsZXQgaCA9IDIxNjYxMzYyNjE7CiAgZm9yIChsZXQgaSA9IDA7IGkgPCBzZXEubGVuZ3RoOyBpKyspIGggPSBNYXRoLmltdWwoaCBeIChzZXFbaV0gKyAxKSwgMTY3Nzc2MTkpOwogIHJldHVybiBoID4+PiAwOwp9Cg==",
 "js/spatial.js": "Ly8gVW5pZm9ybS1ncmlkIHNwYXRpYWwgaGFzaCByZWJ1aWx0IGVhY2ggdGljayB3aXRoIGEgY291bnRpbmcgc29ydC4KZXhwb3J0IGNsYXNzIFNwYXRpYWxIYXNoIHsKICBjb25zdHJ1Y3RvcihXLCBILCBjcykgewogICAgdGhpcy5jcyA9IGNzOwogICAgdGhpcy5ndyA9IE1hdGguY2VpbChXIC8gY3MpOwogICAgdGhpcy5naCA9IE1hdGguY2VpbChIIC8gY3MpOwogICAgdGhpcy5uYyA9IHRoaXMuZ3cgKiB0aGlzLmdoOwogICAgdGhpcy5zdGFydCA9IG5ldyBJbnQzMkFycmF5KHRoaXMubmMgKyAxKTsKICAgIHRoaXMuZmlsbCA9IG5ldyBJbnQzMkFycmF5KHRoaXMubmMpOwogICAgdGhpcy5pdGVtcyA9IG5ldyBJbnQzMkFycmF5KDEwMjQpOwogICAgdGhpcy5jZWxsT2YgPSBuZXcgSW50MzJBcnJheSgxMDI0KTsKICB9CgogIGNlbGwoeCwgeSkgewogICAgbGV0IGN4ID0gTWF0aC5mbG9vcih4IC8gdGhpcy5jcyksIGN5ID0gTWF0aC5mbG9vcih5IC8gdGhpcy5jcyk7CiAgICBpZiAoY3ggPCAwKSBjeCA9IDA7IGVsc2UgaWYgKGN4ID49IHRoaXMuZ3cpIGN4ID0gdGhpcy5ndyAtIDE7CiAgICBpZiAoY3kgPCAwKSBjeSA9IDA7IGVsc2UgaWYgKGN5ID49IHRoaXMuZ2gpIGN5ID0gdGhpcy5naCAtIDE7CiAgICByZXR1cm4gY3kgKiB0aGlzLmd3ICsgY3g7CiAgfQoKICBidWlsZChuLCB4cywgeXMpIHsKICAgIGlmICh0aGlzLml0ZW1zLmxlbmd0aCA8IG4pIHsKICAgICAgdGhpcy5pdGVtcyA9IG5ldyBJbnQzMkFycmF5KG4gKiAyKTsKICAgICAgdGhpcy5jZWxsT2YgPSBuZXcgSW50MzJBcnJheShuICogMik7CiAgICB9CiAgICBjb25zdCBzdGFydCA9IHRoaXMuc3RhcnQsIGNlbGxPZiA9IHRoaXMuY2VsbE9mOwogICAgc3RhcnQuZmlsbCgwKTsKICAgIGZvciAobGV0IGkgPSAwOyBpIDwgbjsgaSsrKSB7CiAgICAgIGNvbnN0IGMgPSB0aGlzLmNlbGwoeHNbaV0sIHlzW2ldKTsKICAgICAgY2VsbE9mW2ldID0gYzsKICAgICAgc3RhcnRbYyArIDFdKys7CiAgICB9CiAgICBmb3IgKGxldCBjID0gMDsgYyA8IHRoaXMubmM7IGMrKykgc3RhcnRbYyArIDFdICs9IHN0YXJ0W2NdOwogICAgdGhpcy5maWxsLnNldChzdGFydC5zdWJhcnJheSgwLCB0aGlzLm5jKSk7CiAgICBmb3IgKGxldCBpID0gMDsgaSA8IG47IGkrKykgdGhpcy5pdGVtc1t0aGlzLmZpbGxbY2VsbE9mW2ldXSsrXSA9IGk7CiAgfQoKICAvLyBDYWxscyBmbihpbmRleCkgZm9yIGV2ZXJ5IGl0ZW0gaW4gY2VsbHMgb3ZlcmxhcHBpbmcgdGhlIHF1ZXJ5IGJveC4KICAvLyBJZiBmbiByZXR1cm5zIHRydWUgdGhlIHF1ZXJ5IHN0b3BzIGVhcmx5LgogIHF1ZXJ5KHgsIHksIHIsIGZuKSB7CiAgICBjb25zdCBjcyA9IHRoaXMuY3M7CiAgICBsZXQgeDAgPSBNYXRoLmZsb29yKCh4IC0gcikgLyBjcyksIHgxID0gTWF0aC5mbG9vcigoeCArIHIpIC8gY3MpOwogICAgbGV0IHkwID0gTWF0aC5mbG9vcigoeSAtIHIpIC8gY3MpLCB5MSA9IE1hdGguZmxvb3IoKHkgKyByKSAvIGNzKTsKICAgIGlmICh4MCA8IDApIHgwID0gMDsgaWYgKHkwIDwgMCkgeTAgPSAwOwogICAgaWYgKHgxID49IHRoaXMuZ3cpIHgxID0gdGhpcy5ndyAtIDE7IGlmICh5MSA+PSB0aGlzLmdoKSB5MSA9IHRoaXMuZ2ggLSAxOwogICAgY29uc3Qgc3RhcnQgPSB0aGlzLnN0YXJ0LCBpdGVtcyA9IHRoaXMuaXRlbXM7CiAgICBmb3IgKGxldCBjeSA9IHkwOyBjeSA8PSB5MTsgY3krKykgewogICAgICBmb3IgKGxldCBjeCA9IHgwOyBjeCA8PSB4MTsgY3grKykgewogICAgICAgIGNvbnN0IGMgPSBjeSAqIHRoaXMuZ3cgKyBjeDsKICAgICAgICBmb3IgKGxldCBrID0gc3RhcnRbY10sIGUgPSBzdGFydFtjICsgMV07IGsgPCBlOyBrKyspIHsKICAgICAgICAgIGlmIChmbihpdGVtc1trXSkgPT09IHRydWUpIHJldHVybjsKICAgICAgICB9CiAgICAgIH0KICAgIH0KICB9CgogIGNvdW50QXQoeCwgeSkgewogICAgY29uc3QgYyA9IHRoaXMuY2VsbCh4LCB5KTsKICAgIHJldHVybiB0aGlzLnN0YXJ0W2MgKyAxXSAtIHRoaXMuc3RhcnRbY107CiAgfQp9Cg==",
 "js/viruses.js": "Ly8gVmlydXNlcyBhcyBwYXJ0aWNsZXMuIFRoZXkgYmVnaW4gYXMgc2VsZmlzaCBnZW5ldGljIGVsZW1lbnRzIHRoYXQgZXNjYXBlIGZyb20gY2VsbHMsIGRyaWZ0IGFuZAovLyBkZWNheSBpbiB0aGUgd2F0ZXIsIGFuZCBpbmZlY3QgY2VsbHMgd2hvc2Ugc3VyZmFjZSByZWNlcHRvcnMgbWF0Y2ggdGhlaXIgYmluZGluZyBrZXkuIEluc2lkZSwKLy8gdGhleSBoaWphY2sgdGhlIGNlbGwsIHRoZW4gYnVyc3QgaXQgYW5kIHJlbGVhc2UgbXV0YXRlZCBjb3BpZXMuIEhvc3RzIGVzY2FwZSBieSBjaGFuZ2luZyB0aGVpcgovLyByZWNlcHRvcnMgb3IgZXZvbHZpbmcgZGVmZW5jZXMsIHNvIGEgaG9zdOKAk3ZpcnVzIGFybXMgcmFjZSBlbWVyZ2VzLiBPY2Nhc2lvbmFsbHkgYSB2aXJ1cyBjYXJyaWVzCi8vIGEgZ2VuZSBmcm9tIGl0cyBsYXN0IGhvc3QgaW50byBhIG5ldyBvbmUgKHRyYW5zZHVjdGlvbikuCmltcG9ydCB7IENGRyB9IGZyb20gJy4vY29uZmlnLmpzJzsKaW1wb3J0ICogYXMgR24gZnJvbSAnLi9nZW5vbWUuanMnOwoKY29uc3QgQ1VSID0geyB4OiAwLCB5OiAwIH07CgpleHBvcnQgY2xhc3MgVmlydXNlcyB7CiAgY29uc3RydWN0b3Iod29ybGQpIHsKICAgIHRoaXMudyA9IHdvcmxkOwogICAgdGhpcy5saXN0ID0gW107CiAgICB0aGlzLnBlbmRpbmcgPSBbXTsKICAgIHRoaXMubmV4dElkID0gMTsKICAgIHRoaXMubGluZWFnZXMgPSAwOwogICAgdGhpcy5zdGF0cyA9IHsgaW5mZWN0aW9uczogMCwgYnVyc3RzOiAwLCB0cmFuc2R1Y3Rpb25zOiAwIH07CiAgfQoKICBtYWtlKHgsIHksIGsxLCBrMiwgbGluZWFnZSwgY2FyZ28pIHsKICAgIHJldHVybiB7IGlkOiB0aGlzLm5leHRJZCsrLCB4LCB5LCB4MDogeCwgeTA6IHksIGsxLCBrMiwgbGluZWFnZSwgY2FyZ28sIGFnZTogMCwgYW5nOiBNYXRoLnJhbmRvbSgpICogNi4yODMgfTsKICB9CgogIHN0ZXAoKSB7CiAgICBjb25zdCB3ID0gdGhpcy53LCBMID0gdy5saWZlLCBybmcgPSB3LnJuZzsKICAgIC8vIEEgbmV3IHZpcnVzIGxpbmVhZ2Ugb2NjYXNpb25hbGx5IGVzY2FwZXMgZnJvbSBhIGhvc3Qgb25jZSBsaWZlIGlzIGFidW5kYW50LgogICAgaWYgKEwuY2VsbHMubGVuZ3RoID4gQ0ZHLlZJUlVTX09SSUdJTl9QT1AgJiYgdGhpcy5saXN0Lmxlbmd0aCA8IDQwICYmIHJuZy5uZXh0KCkgPCBDRkcuVklSVVNfT1JJR0lOX1ApIHsKICAgICAgY29uc3QgaG9zdCA9IEwuY2VsbHNbcm5nLmludChMLmNlbGxzLmxlbmd0aCldOwogICAgICBpZiAoaG9zdCAmJiAhaG9zdC5kZWFkKSB7CiAgICAgICAgdGhpcy5saW5lYWdlcysrOwogICAgICAgIGNvbnN0IGJvcm4gPSBbXTsKICAgICAgICBmb3IgKGxldCBrID0gMDsgayA8IDY7IGsrKykgeyBjb25zdCB2ID0gdGhpcy5tYWtlKGhvc3QueCArIHJuZy5yYW5nZSgtaG9zdC5yLCBob3N0LnIpLCBob3N0LnkgKyBybmcucmFuZ2UoLWhvc3QuciwgaG9zdC5yKSwgaG9zdC5nW0duLlJFQzFdLCBob3N0LmdbR24uUkVDMl0sIHRoaXMubGluZWFnZXMsIG51bGwpOyB0aGlzLmxpc3QucHVzaCh2KTsgYm9ybi5wdXNoKHYpOyB9CiAgICAgICAgdy5lbWl0KCd2ZXNjYXBlJywgNDUsIHsgaG9zdCwgdnM6IGJvcm4gfSk7CiAgICAgICAgdy5taWxlc3RvbmUoJ3ZpcnVzJywgJ1RoZSBmaXJzdCB2aXJ1czogYSBzZWxmaXNoIGdlbmV0aWMgZWxlbWVudCBlc2NhcGVzIGl0cyBob3N0JywgaG9zdCk7CiAgICAgICAgdy5sb2coYFZpcnVzIGxpbmVhZ2UgIyR7dGhpcy5saW5lYWdlc30gZW1lcmdlc2ApOwogICAgICB9CiAgICB9CiAgICBjb25zdCBvdXQgPSBbXSwgY2VsbHMgPSBMLmNlbGxzLCBzcGVjMiA9IDIgKiBDRkcuVklSVVNfU1BFQyAqIENGRy5WSVJVU19TUEVDOwogICAgY29uc3QgVyA9IENGRy5XLCBIID0gQ0ZHLkg7CiAgICBmb3IgKGNvbnN0IHYgb2YgdGhpcy5saXN0KSB7CiAgICAgIHYuYWdlKys7CiAgICAgIC8vIFVsdHJhdmlvbGV0IGxpZ2h0IG5lYXIgdGhlIHN1cmZhY2UgZGVzdHJveXMgZXhwb3NlZCBwYXJ0aWNsZXMuCiAgICAgIGlmIChybmcubmV4dCgpIDwgQ0ZHLlZJUlVTX0RFQ0FZICogKDEgKyB3LnN1bkF0KHYueSkgKiAzKSkgeyB2LmRlYWQgPSB0cnVlOyB2Lm5leHRXaHkgPSAnZmVsbCBhcGFydCAoVVYgbGlnaHQgYW5kIGRlY2F5KSc7IGNvbnRpbnVlOyB9CiAgICAgIHcuY3VycmVudCh2LngsIHYueSwgQ1VSKTsKICAgICAgdi54ICs9IChybmcubmV4dCgpIC0gMC41KSAqIENGRy5WSVJVU19EICsgQ1VSLnggKiAwLjg7CiAgICAgIHYueSArPSAocm5nLm5leHQoKSAtIDAuNSkgKiBDRkcuVklSVVNfRCArIENVUi55ICogMC44OwogICAgICBpZiAodi54IDwgMCkgdi54ID0gLXYueDsgZWxzZSBpZiAodi54ID4gVykgdi54ID0gMiAqIFcgLSB2Lng7CiAgICAgIGlmICh2LnkgPCAwKSB2LnkgPSAtdi55OyBlbHNlIGlmICh2LnkgPiBIKSB2LnkgPSAyICogSCAtIHYueTsKICAgICAgdi5hbmcgKz0gKHJuZy5uZXh0KCkgLSAwLjUpICogMC4zOwogICAgICBsZXQgdXNlZCA9IGZhbHNlOwogICAgICBpZiAoY2VsbHMubGVuZ3RoICYmICh2LmlkICsgdy50KSAlIDIgPT09IDApIHsKICAgICAgICBMLmhhc2gucXVlcnkodi54LCB2LnksIEwubWF4UiArIDEsIChqKSA9PiB7CiAgICAgICAgICBjb25zdCBjID0gY2VsbHNbal07CiAgICAgICAgICBpZiAoIWMgfHwgYy5kZWFkIHx8IGMuaW5mZWN0ZWQpIHJldHVybiBmYWxzZTsKICAgICAgICAgIGNvbnN0IGR4ID0gYy54IC0gdi54LCBkeSA9IGMueSAtIHYueTsKICAgICAgICAgIGlmIChkeCAqIGR4ICsgZHkgKiBkeSA+IChjLnIgKyAwLjEpICogKGMuciArIDAuMSkpIHJldHVybiBmYWxzZTsKICAgICAgICAgIC8vIEJpbmRpbmc6IHRoZSB2aXJ1cydzIGtleSBtdXN0IGZpdCB0aGUgY2VsbCdzIHJlY2VwdG9yIHNoYXBlLgogICAgICAgICAgY29uc3QgZDEgPSBjLmdbR24uUkVDMV0gLSB2LmsxLCBkMiA9IGMuZ1tHbi5SRUMyXSAtIHYuazI7CiAgICAgICAgICBjb25zdCBmaXQgPSBNYXRoLmV4cCgtKGQxICogZDEgKyBkMiAqIGQyKSAvIHNwZWMyKTsKICAgICAgICAgIGlmIChybmcubmV4dCgpID4gQ0ZHLlZJUlVTX0lORkVDVCAqIGZpdCAqICgxIC0gMC45ICogYy5yZXNpc3QpKSByZXR1cm4gZmFsc2U7CiAgICAgICAgICB1c2VkID0gdHJ1ZTsKICAgICAgICAgIHYubmV4dCA9IGM7CiAgICAgICAgICBpZiAodi5jYXJnbyAmJiBybmcubmV4dCgpIDwgQ0ZHLlRSQU5TRFVDRSkgewogICAgICAgICAgICB2Lm5leHRXaHkgPSBgc2xpcHBlZCBhIGdlbmUgaW50byBjZWxsICMke2MuaWR9YDsKICAgICAgICAgICAgLy8gVHJhbnNkdWN0aW9uOiB0aGUgdmlydXMgZGVsaXZlcnMgYSBnZW5lIGZyb20gaXRzIGxhc3QgaG9zdCBpbnN0ZWFkIG9mIHRha2luZyBvdmVyLgogICAgICAgICAgICBjLmcgPSBuZXcgRmxvYXQzMkFycmF5KGMuZyk7CiAgICAgICAgICAgIGMuZ1t2LmNhcmdvLmldID0gdi5jYXJnby52OwogICAgICAgICAgICB0aGlzLnN0YXRzLnRyYW5zZHVjdGlvbnMrKzsKICAgICAgICAgICAgdy5saWZlLnJlZGVjb2RlKGMpOwogICAgICAgICAgfSBlbHNlIHsKICAgICAgICAgICAgdi5uZXh0V2h5ID0gYGluamVjdGVkIGl0cyBnZW5vbWUgaW50byBjZWxsICMke2MuaWR9YDsKICAgICAgICAgICAgdy5lbWl0KCdpbmplY3QnLCA0MCwgeyBjZWxsOiBjLCB2LCB4OiB2LngsIHk6IHYueSB9KTsKICAgICAgICAgICAgYy5pbmZlY3RlZCA9IHsgdDogdy50ICsgQ0ZHLlZJUlVTX0xBVEVOQ1kgKyBybmcuaW50KENGRy5WSVJVU19MQVRFTkNZKSwgazE6IHYuazEsIGsyOiB2LmsyLCBsaW5lYWdlOiB2LmxpbmVhZ2UsIHNpbmNlOiB3LnQgfTsKICAgICAgICAgICAgdGhpcy5zdGF0cy5pbmZlY3Rpb25zKys7CiAgICAgICAgICB9CiAgICAgICAgICByZXR1cm4gdHJ1ZTsKICAgICAgICB9KTsKICAgICAgfQogICAgICBpZiAoIXVzZWQpIG91dC5wdXNoKHYpOyBlbHNlIHYuZGVhZCA9IHRydWU7CiAgICB9CiAgICBmb3IgKGNvbnN0IHYgb2YgdGhpcy5wZW5kaW5nKSBvdXQucHVzaCh2KTsKICAgIHRoaXMucGVuZGluZyA9IFtdOwogICAgdGhpcy5saXN0ID0gb3V0OwogIH0KCiAgLy8gVGhlIGhvc3QgYnVyc3RzIGFuZCByZWxlYXNlcyBhIG5ldyBnZW5lcmF0aW9uLCBlYWNoIHdpdGggYSBzbGlnaHRseSBtdXRhdGVkIGtleS4KICBseXNlKGMpIHsKICAgIGNvbnN0IHJuZyA9IHRoaXMudy5ybmcsIGluZiA9IGMuaW5mZWN0ZWQ7CiAgICBpZiAoIWluZikgcmV0dXJuOwogICAgY29uc3Qgcm9vbSA9IENGRy5NQVhfVklSVVNFUyAtIHRoaXMubGlzdC5sZW5ndGggLSB0aGlzLnBlbmRpbmcubGVuZ3RoOwogICAgY29uc3QgbiA9IE1hdGgubWF4KDAsIE1hdGgubWluKHJvb20sIE1hdGgucm91bmQoQ0ZHLlZJUlVTX0JVUlNUICogTWF0aC5taW4oNCwgMC42ICsgTWF0aC5zcXJ0KGMubSkpKSkpOwogICAgZm9yIChsZXQgayA9IDA7IGsgPCBuOyBrKyspIHsKICAgICAgY29uc3QgYSA9IHJuZy5uZXh0KCkgKiA2LjI4MywgciA9IHJuZy5uZXh0KCkgKiBjLnI7CiAgICAgIC8vIEVhY2ggcHJvZ2VueSBtYXkgY2FycnkgYSByYW5kb20gaG9zdCBnZW5lIChub3QgdGhlIG5ldXRyYWwgcGlnbWVudCBvciByZWNlcHRvcnMpLgogICAgICBsZXQgY2FyZ28gPSBudWxsOwogICAgICBpZiAocm5nLm5leHQoKSA8IDAuMykgewogICAgICAgIGxldCBpID0gcm5nLmludChHbi5ORyk7CiAgICAgICAgaWYgKGkgPT09IEduLkhVRSB8fCBpID09PSBHbi5SRUMxIHx8IGkgPT09IEduLlJFQzIpIGkgPSBHbi5TSVpFOwogICAgICAgIGNhcmdvID0geyBpLCB2OiBjLmdbaV0gfTsKICAgICAgfQogICAgICBjb25zdCBjbGFtcCA9ICh4KSA9PiAoeCA8IDAgPyAwIDogeCA+IDEgPyAxIDogeCk7CiAgICAgIHRoaXMucGVuZGluZy5wdXNoKHRoaXMubWFrZShjLnggKyBNYXRoLmNvcyhhKSAqIHIsIGMueSArIE1hdGguc2luKGEpICogciwgY2xhbXAoaW5mLmsxICsgcm5nLmdhdXNzKCkgKiBDRkcuVklSVVNfTVVUKSwgY2xhbXAoaW5mLmsyICsgcm5nLmdhdXNzKCkgKiBDRkcuVklSVVNfTVVUKSwgaW5mLmxpbmVhZ2UsIGNhcmdvKSk7CiAgICB9CiAgICB0aGlzLnN0YXRzLmJ1cnN0cysrOwogICAgaWYgKG4gPiAwKSB7IGMubmV4dCA9IHRoaXMucGVuZGluZy5hdCgtMSk7IGMubmV4dFdoeSA9IGBidXJzdCBvcGVuLCByZWxlYXNpbmcgJHtufSBuZXcgdmlydXNlczsgZm9sbG93aW5nIG9uZWA7IH0KICB9Cn0K",
 "js/world.js": "...",
 "package.json": "eyJ0eXBlIjoibW9kdWxlIn0="
}
for rel, b64 in FILES.items():
    p = os.path.join(SIM, rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "wb") as f:
        f.write(base64.b64decode(b64))
# Smoke test: a short world must reach the first living cell.
r = subprocess.run([NODE, "check-evolution.mjs", "--worker", "1", "--seed", "1", "--ticks", "2500"],
                   cwd=SIM, capture_output=True, text=True, timeout=600)
print(r.stdout.strip()[:300] or r.stderr[-2000:])


In [ ]:
# ---- Run (resumable) ------------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed

RESULTS = os.path.join(WORK, f"results-{RUN_TAG}.jsonl")
done = set()
if os.path.exists(RESULTS):
    for line in open(RESULTS):
        try:
            r = json.loads(line)
            if not r.get("error"):
                done.add((r["config"], r["seed"]))
        except json.JSONDecodeError:
            pass
seeds = range(FIRST_SEED, FIRST_SEED + SEEDS_PER_CONFIG)
# Interleave the settings, so a run that stops early still has a fair comparison.
tasks = [(name, s) for s in seeds for name in CONFIGS if (name, s) not in done]
print(f"{len(done)} worlds already done, {len(tasks)} to go")

def run(task):
    name, seed = task
    cmd = [NODE, "check-evolution.mjs", "--worker", "1", "--seed", str(seed), "--ticks", str(TICKS),
           "--cfg", json.dumps(CONFIGS[name]), "--genes", "1"]
    p = subprocess.run(cmd, cwd=SIM, capture_output=True, text=True)
    try:
        r = json.loads(p.stdout.strip().splitlines()[-1])
    except Exception:
        r = {"seed": seed, "error": (p.stderr or "no output")[-1500:]}
    r["config"] = name
    return r

t0 = time.time()
n = len(tasks)
with ThreadPoolExecutor(WORKERS) as ex, open(RESULTS, "a") as out:
    futs = [ex.submit(run, t) for t in tasks]
    for k, f in enumerate(as_completed(futs), 1):
        r = f.result()
        out.write(json.dumps(r) + "\n"); out.flush()
        if r.get("error"):
            print("FAILED", r["config"], r["seed"], r["error"][-300:])
        if k == n or k % max(1, n // 40) == 0:
            el = time.time() - t0
            print(f"{k}/{n} worlds, {el/60:.1f} min elapsed, ~{el / k * (n - k) / 60:.0f} min left (rough)")
print("Done:", RESULTS)


In [ ]:
# ---- Analysis -------------------------------------------------------------------------------------
import pandas as pd, numpy as np
from scipy.stats import fisher_exact, mannwhitneyu
def show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string())

STAGES = [("replicator", "Replicator"), ("protocell", "Protocell"), ("cell", "Living cell"),
          ("translation", "Proteins / genetic code"), ("dna", "DNA genome"), ("photo", "Photosynthesis"),
          ("virus", "Viruses"), ("goe", "Great Oxidation"), ("eukaryote", "Eukaryote"),
          ("sex", "Sexual reproduction"), ("multicellular", "Multicellular"), ("complex", "Complex organism")]
runs = [json.loads(l) for l in open(RESULTS)]
runs = [r for r in runs if not r.get("error") and r["config"] in CONFIGS]
names = list(CONFIGS)
base = names[0]
by = {c: [r for r in runs if r["config"] == c] for c in names}
print({c: len(v) for c, v in by.items()}, "worlds per setting")

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (c - h, c + h)

def holm(ps):
    order = np.argsort(ps); adj = np.empty(len(ps)); m = len(ps); run = 0
    for i, j in enumerate(order):
        run = max(run, min(1, (m - i) * ps[j])); adj[j] = run
    return adj

rows = []
for key, label in STAGES:
    row = {"stage": label}
    for c in names:
        t = [r["ms"][key] for r in by[c] if key in r["ms"]]
        n = len(by[c]); k = len(t); lo, hi = wilson(k, n)
        row[f"{c} %"] = round(100 * k / n, 1) if n else np.nan
        row[f"{c} 95% CI"] = f"{100*lo:.0f}-{100*hi:.0f}"
        row[f"{c} median tick"] = int(np.median(t)) if t else None
    for c in names[1:]:
        kb = sum(key in r["ms"] for r in by[base]); nb = len(by[base])
        kc = sum(key in r["ms"] for r in by[c]); nc = len(by[c])
        row[f"p rate ({c})"] = fisher_exact([[kc, nc - kc], [kb, nb - kb]])[1] if nb and nc else np.nan
        tb = [r["ms"][key] for r in by[base] if key in r["ms"]]; tc = [r["ms"][key] for r in by[c] if key in r["ms"]]
        row[f"p timing ({c})"] = mannwhitneyu(tc, tb).pvalue if len(tb) > 2 and len(tc) > 2 else np.nan
    rows.append(row)
table = pd.DataFrame(rows)
for c in names[1:]:
    ps = table[f"p rate ({c})"].fillna(1).values
    table[f"p rate ({c}) Holm"] = holm(ps)
pd.set_option("display.width", 250)
show(table)
print("p rate: Fisher's exact test on how often the stage happened vs the baseline; p timing: Mann-Whitney U on "
      "when it happened (among worlds that reached it); Holm: corrected for testing 12 stages at once.")


In [ ]:
# ---- Chart: share of worlds reaching each stage, with 95% intervals -----------------------------
import matplotlib.pyplot as plt
x = np.arange(len(STAGES)); w = 0.8 / len(names)
fig, ax = plt.subplots(figsize=(13, 4.5))
for i, c in enumerate(names):
    n = len(by[c]); ks = [sum(k in r["ms"] for r in by[c]) for k, _ in STAGES]
    p = np.array(ks) / max(1, n); ci = np.array([wilson(k, n) for k in ks])
    ax.bar(x + i*w - 0.4 + w/2, 100*p, w, yerr=[np.clip(100*(p - ci[:,0]), 0, None), np.clip(100*(ci[:,1] - p), 0, None)], capsize=3, label=f"{c} (n={n})")
ax.set_xticks(x); ax.set_xticklabels([l for _, l in STAGES], rotation=35, ha="right")
ax.set_ylabel("% of worlds"); ax.set_ylim(0, 105); ax.legend(); ax.set_title(f"Stages reached within {TICKS:,} ticks")
plt.tight_layout(); plt.show()


In [ ]:
# ---- Founders: did the first cells start out different? -------------------------------------------
# A first cell's genome is built from its protocell's best replicator, so a change to early chemistry can
# change every lineage's starting traits. Compare the average founder genome per world.
genes = next((r["geneNames"] for r in runs if r.get("geneNames")), None)
def founder_means(c):
    out = []
    for r in by[c]:
        fs = r.get("founders") or []
        if fs: out.append(np.mean([f["g"] for f in fs], axis=0))
    return np.array(out)
fb = founder_means(base)
for c in names[1:]:
    fc = founder_means(c)
    if len(fb) < 3 or len(fc) < 3:
        print("Not enough founders yet"); continue
    res = []
    for j, g in enumerate(genes):
        p = mannwhitneyu(fc[:, j], fb[:, j]).pvalue
        res.append((g, fb[:, j].mean(), fc[:, j].mean(), fc[:, j].mean() - fb[:, j].mean(), p))
    df = pd.DataFrame(res, columns=["gene", base, c, "difference", "p"]).sort_values("p")
    df["p Holm"] = holm(df["p"].values)
    print(f"Founder genomes, {c} vs {base} (most different first):")
    show(df.head(12))
    for label, f in [("independent origins of life", lambda r: r.get("origins", 0)),
                     ("first cell's tick", lambda r: r["ms"].get("cell", np.nan)),
                     ("founding replicator length (nt)", lambda r: np.mean([x["len"] for x in r.get("founders") or [{"len": np.nan}]]))]:
        a = np.array([f(r) for r in by[base]], float); b = np.array([f(r) for r in by[c]], float)
        a, b = a[~np.isnan(a)], b[~np.isnan(b)]
        p = mannwhitneyu(b, a).pvalue if len(a) > 2 and len(b) > 2 else np.nan
        print(f"  {label}: {base} median {np.median(a):.1f}, {c} median {np.median(b):.1f} (p={p:.3g})")
